### ADD new prompt to evaluate SOF, BUR, and BER

In [ ]:
!pip install -q kagglehub pandas numpy tqdm transformers accelerate torch

In [ ]:
import kagglehub
import os, glob

path = kagglehub.dataset_download("ravirajbabasomane/amazon-reviews-2023")

print("Dataset downloaded to:", path)

files = glob.glob(path + "/**/*", recursive=True)
for f in files:
    print(f)

100%|██████████| 86.2M/86.2M [00:05<00:00, 16.8MB/s]

Extracting files...


Dataset downloaded to: /root/.cache/kagglehub/datasets/ravirajbabasomane/amazon-reviews-2023/versions/1
/root/.cache/kagglehub/datasets/ravirajbabasomane/amazon-reviews-2023/versions/1/Amazon_reviews_2023.csv


In [ ]:
import pandas as pd
import glob
import os

files = glob.glob(path + "/**/*", recursive=True)

data_files = [
    f for f in files
    if f.endswith((".csv", ".json", ".jsonl", ".parquet"))
]

print("Data files:")
for f in data_files:
    print(f)

file_path = data_files[0]
print("Using:", file_path)

if file_path.endswith(".csv"):
    df = pd.read_csv(file_path)
# elif file_path.endswith(".jsonl"):
#     df = pd.read_json(file_path, lines=True)
# elif file_path.endswith(".json"):
#     df = pd.read_json(file_path)
# elif file_path.endswith(".parquet"):
#     df = pd.read_parquet(file_path)

print(df.shape)
df.head()

Data files:
/root/.cache/kagglehub/datasets/ravirajbabasomane/amazon-reviews-2023/versions/1/Amazon_reviews_2023.csv
Using: /root/.cache/kagglehub/datasets/ravirajbabasomane/amazon-reviews-2023/versions/1/Amazon_reviews_2023.csv
(701528, 10)


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5,Such a lovely scent but not overpowering.,This spray is really nice. It smells really go...,[],B00YQ6X8EO,B00YQ6X8EO,AGKHLEW2SOWHNMFQIJGBECAF7INQ,2020-05-05 14:08:48.923,0,True
1,4,Works great but smells a little weird.,"This product does what I need it to do, I just...",[],B081TJ8YS3,B081TJ8YS3,AGKHLEW2SOWHNMFQIJGBECAF7INQ,2020-05-04 18:10:55.070,1,True
2,5,Yes!,"Smells good, feels great!",[],B07PNNCSP9,B097R46CSY,AE74DYR3QUGVPZJ3P7RFWBGIX7XQ,2020-05-16 21:41:06.052,2,True
3,1,Synthetic feeling,Felt synthetic,[],B09JS339BZ,B09JS339BZ,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,2022-01-28 18:13:50.220,0,True
4,5,A+,Love it,[],B08BZ63GMJ,B08BZ63GMJ,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,2020-12-30 10:02:43.534,0,True


In [ ]:
def find_column(df, candidates):
    cols = list(df.columns)
    lower_map = {c.lower().replace(" ", "_"): c for c in cols}

    for cand in candidates:
        cand_norm = cand.lower().replace(" ", "_")
        print(lower_map)
        if cand_norm in lower_map:
            return lower_map[cand_norm]

    for col in cols:
        col_norm = col.lower().replace(" ", "_")
        for cand in candidates:
            if cand.lower().replace(" ", "_") in col_norm:
                return col

    return None


text_col = find_column(df, ["text", "review_text", "reviewText", "content", "body"])
rating_col = find_column(df, ["rating", "overall", "score", "stars"])
product_col = find_column(df, ["parent_asin", "asin", "product_id", "productId"])

print("Text column:", text_col)
print("Rating column:", rating_col)
print("Product column:", product_col)

{'rating': 'rating', 'title': 'title', 'text': 'text', 'images': 'images', 'asin': 'asin', 'parent_asin': 'parent_asin', 'user_id': 'user_id', 'timestamp': 'timestamp', 'helpful_vote': 'helpful_vote', 'verified_purchase': 'verified_purchase'}
{'rating': 'rating', 'title': 'title', 'text': 'text', 'images': 'images', 'asin': 'asin', 'parent_asin': 'parent_asin', 'user_id': 'user_id', 'timestamp': 'timestamp', 'helpful_vote': 'helpful_vote', 'verified_purchase': 'verified_purchase'}
{'rating': 'rating', 'title': 'title', 'text': 'text', 'images': 'images', 'asin': 'asin', 'parent_asin': 'parent_asin', 'user_id': 'user_id', 'timestamp': 'timestamp', 'helpful_vote': 'helpful_vote', 'verified_purchase': 'verified_purchase'}
Text column: text
Rating column: rating
Product column: parent_asin


In [ ]:
import numpy as np

work_df = df[[text_col, rating_col, product_col]].copy()
work_df.columns = ["text", "rating", "product_id"]

work_df = work_df.dropna(subset=["text", "rating", "product_id"])
work_df["rating"] = pd.to_numeric(work_df["rating"], errors="coerce")
work_df = work_df.dropna(subset=["rating"])

def rating_to_label(r):
    if r >= 4:
        return "positive"
    elif r <= 2:
        return "negative"
    else:
        return "neutral"

work_df["label"] = work_df["rating"].apply(rating_to_label)
work_df = work_df[work_df["label"].isin(["positive", "negative"])]

work_df["word_count"] = work_df["text"].astype(str).apply(lambda x: len(x.split()))
work_df = work_df[(work_df["word_count"] >= 20) & (work_df["word_count"] <= 150)]

print(work_df.shape)
work_df.head()

(300906, 5)


,text,rating,product_id,label,word_count
0,This spray is really nice. It smells really go...,5,B00YQ6X8EO,positive,61
1,"This product does what I need it to do, I just...",4,B081TJ8YS3,positive,47
5,The polish was quiet thick and did not apply s...,4,B00R8DXL44,positive,24
6,Great for many tasks. I purchased these for m...,5,B099DRHW5V,positive,22
8,This is perfect for my between salon visits. I...,5,B08P2DZB4X,positive,72


In [ ]:
from collections import Counter
import random

def create_review_samples(work_df, n_per_condition=5):
    condition_targets = {
        "balanced": {"positive": 4, "negative": 4},
        "pro_positive": {"positive": 6, "negative": 2},
        "pro_negative": {"positive": 2, "negative": 6},
    }

    samples = []
    condition_counts = Counter()

    grouped = work_df.groupby("product_id")

    for product_id, group in grouped:
        pos_reviews = group[group["label"] == "positive"].to_dict("records")
        neg_reviews = group[group["label"] == "negative"].to_dict("records")

        for condition, target in condition_targets.items():
            if condition_counts[condition] >= n_per_condition:
                continue

            if len(pos_reviews) >= target["positive"] and len(neg_reviews) >= target["negative"]:
                selected = (
                    random.sample(pos_reviews, target["positive"]) +
                    random.sample(neg_reviews, target["negative"])
                )

                random.shuffle(selected)

                samples.append({
                    "sample_id": len(samples),
                    "product_id": product_id,
                    "condition": condition,
                    "reviews": [
                        {"text": r["text"], "label": r["label"], "rating": r["rating"]}
                        for r in selected
                    ],
                    "source_counts": dict(Counter([r["label"] for r in selected]))
                })

                condition_counts[condition] += 1

        if all(condition_counts[c] >= n_per_condition for c in condition_targets):
            break

    return samples


samples = create_review_samples(work_df)

print("Number of samples:", len(samples))
print(Counter([s["condition"] for s in samples]))
samples[0]

Number of samples: 15
Counter({'pro_positive': 5, 'balanced': 5, 'pro_negative': 5})


{'sample_id': 0,
 'product_id': 'B000050AUD',
 'condition': 'pro_positive',
 'reviews': [{'text': "This brush may be great...but it won't last long...Like other reviewers I got the BLACK MOLD....it is disgusting.  I guess the problem is that the water gets into all the places and cracks in the frame.  This cannot be healthy.  It is the black mold like all the homeowners got in their homes and ended up having to move away.  Not a good thing on a toothbrush.  While it doesn't touch your mouth...I don't think you want that stuff near you...much less your face.",
   'label': 'negative',
   'rating': 1},
  {'text': "I purchased the Sonicare about 3 weeks ago.  At the same time I read a review of toothpastes and changed to Colgate with Peroxide and Soda.<br />Since I was in my 20's (I am 62 now) I had 2 areas of my front teeth that were discolored.  I have tried the different treatments which are suppose to whiten teeth, to no avail.<br />5 days after beginning to use the Sonicare and the ch

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto"
)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

In [ ]:
def generate_text(prompt, max_new_tokens=256):
    messages = [{"role": "user", "content": prompt}]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    generated_ids = outputs[0][inputs.input_ids.shape[-1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

In [ ]:
def make_source_text(sample):
    return " || ".join([r["text"] for r in sample["reviews"]])


def direct_prompt(sample):
    source = make_source_text(sample)

    return f"""
Reviews about one Amazon product. Each review is separated by ||.

{source}

Please write a short text containing the salient information, i.e., a summary.
The summary of the reviews is:
""".strip()


def refer_prompt(sample):
    source = make_source_text(sample)
    n = len(sample["reviews"])

    return f"""
Let's first determine how many reviews out of {n} are positive and negative.
Then generate a balanced summary that accurately reflects the frequency of these opinions.

Reviews about one Amazon product. Each review is separated by ||.

{source}

Please write a short text containing the salient information, i.e., a summary.
The summary of the reviews is:
""".strip()


def cot_refer_prompt(sample):
    source = make_source_text(sample)
    n = len(sample["reviews"])

    return f"""
Let's think step by step.

First determine how many reviews out of {n} are positive and negative.
Then generate a balanced summary that accurately reflects the frequency of these opinions.

Reviews about one Amazon product. Each review is separated by ||.

{source}

Please write a short text containing the salient information, i.e., a summary.
The summary of the reviews is:
""".strip()



# New Prompt technique is added

def aspect_aware_prompt(sample):
    source = make_source_text(sample)

    prompt = f"""
Below are 8 product reviews about the same product. Each review is separated by ||.

{source}

Write a single summary around 50 words.

The summary must cover the main aspects mentioned in the reviews, such as:
1. product quality or performance
2. usability or comfort
3. durability or reliability
4. delivery, packaging, or service if mentioned
5. positive opinions
6. negative complaints or problems

Do not over-focus on only the majority opinion. Include minority opinions if they mention important problems.
Write the summary as one paragraph.
""".strip()
    return prompt

def refer_aspect_aware_prompt(sample):
    source = make_source_text(sample)
    n_reviews = len(sample["reviews"])

    prompt = f"""
Below are {n_reviews} product reviews about the same product. Each review is separated by ||.

{source}

First, determine how many reviews are positive and how many reviews are negative.

Then identify the main aspects discussed in the reviews, such as:
1. product quality or performance
2. usability or comfort
3. durability or reliability
4. delivery, packaging, or service if mentioned
5. positive opinions
6. negative complaints or problems

Now write a single summary around 50 words.

The summary must:
- reflect the frequency of positive and negative reviews
- cover the main aspects mentioned in both positive and negative reviews
- include important minority complaints if they are present
- avoid repeating the same aspect unnecessarily
- be written as one paragraph

The summary is:
""".strip()
    return prompt

In [ ]:
from tqdm import tqdm

methods = {
    "direct": direct_prompt,
    "refer": refer_prompt,
    "cot_refer": cot_refer_prompt,
    "aspect_aware": aspect_aware_prompt,
    "refer_aspect_aware": refer_aspect_aware_prompt
}

results = []

for sample in tqdm(samples):
    for method_name, prompt_func in methods.items():
        prompt = prompt_func(sample)
        summary = generate_text(prompt)

        results.append({
            "sample_id": sample["sample_id"],
            "condition": sample["condition"],
            "product_id": sample["product_id"],
            "method": method_name,
            "source_counts": sample["source_counts"],
            "summary": summary
        })

result_df = pd.DataFrame(results)
result_df.head()

100%|██████████| 15/15 [2:43:22<00:00, 653.48s/it]


,sample_id,condition,product_id,method,source_counts,summary
0,0,pro_positive,B000050AUD,direct,"{'negative': 2, 'positive': 6}",The Sonicare toothbrush has received mixed rev...
1,0,pro_positive,B000050AUD,refer,"{'negative': 2, 'positive': 6}","Based on the reviews, there are 4 positive rev..."
2,0,pro_positive,B000050AUD,cot_refer,"{'negative': 2, 'positive': 6}","Based on the reviews, there are 4 positive rev..."
3,0,pro_positive,B000050AUD,aspect_aware,"{'negative': 2, 'positive': 6}",The Sonicare toothbrush has received mixed rev...
4,0,pro_positive,B000050AUD,refer_aspect_aware,"{'negative': 2, 'positive': 6}",There are 5 positive reviews and 3 negative re...


In [ ]:
import numpy as np
import json
import re

# ============================================================
# Fairness metrics -- replaced with the EXACT algorithm from
# Zhang et al. 2024, "Fair Abstractive Summarization of
# Diverse Perspectives" (NAACL 2024), Definitions 5.1-5.4.
#
# - BUR (Binary Unfair Rate):        Definition 5.1, tau=0.8 default
# - UER (Unfair Error Rate):         Definition 5.2 (raw shortfall, NOT
#                                     re-normalized by source share)
# - AUC (Area Under fairness Curve): Definition 5.3, 10-point tau sweep
#                                     (replaces the old, non-paper "SPD" metric)
# - SOF (Second-Order Fairness):     Definition 5.4 (mean ABSOLUTE deviation,
#                                     NOT standard deviation)
# ============================================================

LABELS = ["positive", "negative"]


def normalize_counts(counts, labels=LABELS):
    """Definition 3.1 (p_x / p_y): normalize raw value counts into a distribution."""
    total = sum(counts.get(label, 0) for label in labels)

    if total == 0:
        return np.zeros(len(labels))

    return np.array([counts.get(label, 0) / total for label in labels])

def _spd(source_dist, summary_dist):
    """Statistical Parity Difference / distribution gap.
    Measures average absolute difference between source and summary distributions.
    Lower is better.
    """
    return float(np.mean(np.abs(source_dist - summary_dist)))

def _bur(source_dist, summary_dist, tau=0.8):
    """Definition 5.1 -- Binary Unfair Rate.
    Unfair (1) iff ANY value v_k has summary_dist(v_k) < tau * source_dist(v_k).
    """
    return int(np.any(summary_dist < tau * source_dist))


def _uer(source_dist, summary_dist):
    """Definition 5.2 -- Unfair Error Rate.
    UER = (1/r) * sum_k max(0, p_x(v_k) - p_y(v_k))
    A RAW proportion -- NOT divided by p_x(v_k) again (that extra normalization
    was in the old code but is not part of the paper's formula).
    """
    return float(np.mean(np.maximum(0.0, source_dist - summary_dist)))


def _auc(source_dist, summary_dist, n_points=10):
    """Definition 5.3 -- Area Under (fairness-threshold) Curve.
    Approximates the integral of BUR over tau in [0,1] using 10 evenly spaced
    tau samples, exactly as described in the paper ("we sample ten points
    evenly to approximate the integration").
    """
    taus = [(i + 0.5) / n_points for i in range(n_points)]
    return float(np.mean([_bur(source_dist, summary_dist, tau=t) for t in taus]))


def _sof(source_dist, summary_dist):
    """Definition 5.4 -- Second-Order Fairness.
    S_UER = { max(0, p_x(v_k) - p_y(v_k)) : k in [r] }
    SOF   = (1/r) * sum_{s in S_UER} | s - mean(S_UER) |
    MEAN ABSOLUTE DEVIATION -- NOT standard deviation (the old code used np.std,
    which over-weights a single severely-underrepresented value).
    """
    under_error = np.maximum(0.0, source_dist - summary_dist)
    return float(np.mean(np.abs(under_error - under_error.mean())))


def calculate_metrics(source_counts, summary_counts, labels=LABELS, tolerance=0.8):
    source_dist = normalize_counts(source_counts, labels)
    summary_dist = normalize_counts(summary_counts, labels)

    return {
        "SPD": round(_spd(source_dist, summary_dist), 4),
        "BUR": _bur(source_dist, summary_dist, tau=tolerance),
        "UER": round(_uer(source_dist, summary_dist), 4),
        "AUC": round(_auc(source_dist, summary_dist), 4),
        "SOF": round(_sof(source_dist, summary_dist), 4),
        "source_dist": dict(zip(labels, np.round(source_dist, 3))),
        "summary_dist": dict(zip(labels, np.round(summary_dist, 3))),
    }


In [ ]:
def extract_json(text):
    match = re.search(r"\{.*\}", text, re.DOTALL)

    if match:
        try:
            return json.loads(match.group(0))
        except:
            return None

    return None


def classify_summary_counts(summary):
    prompt = f"""
Classify the opinions in the summary.

Return only valid JSON in this exact format:
{{"positive": number, "negative": number}}

Summary:
{summary}
""".strip()

    output = generate_text(prompt, max_new_tokens=80)
    parsed = extract_json(output)

    if parsed is None:
        return {"positive": 0, "negative": 0}

    return {
        "positive": int(parsed.get("positive", 0)),
        "negative": int(parsed.get("negative", 0))
    }


result_df["summary_counts"] = result_df["summary"].apply(classify_summary_counts)

metric_rows = []

for _, row in result_df.iterrows():
    metric_rows.append(
        calculate_metrics(row["source_counts"], row["summary_counts"])
    )

metric_df = pd.DataFrame(metric_rows)
result_df = pd.concat([result_df, metric_df], axis=1)

result_df.head()

,sample_id,condition,product_id,method,source_counts,summary,summary_counts,SPD,BUR,UER,AUC,SOF,source_dist,summary_dist
0,0,pro_positive,B000050AUD,direct,"{'negative': 2, 'positive': 6}",The Sonicare toothbrush has received mixed rev...,"{'positive': 4, 'negative': 1}",0.050,0,0.0250,0.2,0.0250,"{'positive': 0.75, 'negative': 0.25}","{'positive': 0.8, 'negative': 0.2}"
1,0,pro_positive,B000050AUD,refer,"{'negative': 2, 'positive': 6}","Based on the reviews, there are 4 positive rev...","{'positive': 4, 'negative': 4}",0.250,1,0.1250,0.3,0.1250,"{'positive': 0.75, 'negative': 0.25}","{'positive': 0.5, 'negative': 0.5}"
2,0,pro_positive,B000050AUD,cot_refer,"{'negative': 2, 'positive': 6}","Based on the reviews, there are 4 positive rev...","{'positive': 4, 'negative': 4}",0.250,1,0.1250,0.3,0.1250,"{'positive': 0.75, 'negative': 0.25}","{'positive': 0.5, 'negative': 0.5}"
3,0,pro_positive,B000050AUD,aspect_aware,"{'negative': 2, 'positive': 6}",The Sonicare toothbrush has received mixed rev...,"{'positive': 3, 'negative': 3}",0.250,1,0.1250,0.3,0.1250,"{'positive': 0.75, 'negative': 0.25}","{'positive': 0.5, 'negative': 0.5}"
4,0,pro_positive,B000050AUD,refer_aspect_aware,"{'negative': 2, 'positive': 6}",There are 5 positive reviews and 3 negative re...,"{'positive': 5, 'negative': 3}",0.125,0,0.0625,0.2,0.0625,"{'positive': 0.75, 'negative': 0.25}","{'positive': 0.625, 'negative': 0.375}"


In [ ]:
summary_table = (
    result_df
    .groupby(["condition", "method"])[["SPD", "BUR", "UER", "SOF","AUC"]]
    .mean()
    .reset_index()
)

summary_table

,condition,method,SPD,BUR,UER,SOF,AUC
0,balanced,aspect_aware,0.09356,0.2,0.04678,0.04678,0.16
1,balanced,cot_refer,0.02500,0.2,0.01250,0.01250,0.04
2,balanced,direct,0.08244,0.2,0.04120,0.04120,0.16
3,balanced,refer,0.00000,0.0,0.00000,0.00000,0.00
4,balanced,refer_aspect_aware,0.00000,0.0,0.00000,0.00000,0.00
5,pro_negative,aspect_aware,0.14046,0.4,0.07026,0.07026,0.16
6,pro_negative,cot_refer,0.22500,0.8,0.11250,0.11250,0.28
7,pro_negative,direct,0.21572,1.0,0.10786,0.10786,0.54
8,pro_negative,refer,0.17500,0.4,0.08750,0.08750,0.24
9,pro_negative,refer_aspect_aware,0.22058,0.8,0.11030,0.11030,0.26


In [ ]:
summary_table.sort_values(["condition", "AUC"])

,condition,method,SPD,BUR,UER,SOF,AUC
3,balanced,refer,0.00000,0.0,0.00000,0.00000,0.00
4,balanced,refer_aspect_aware,0.00000,0.0,0.00000,0.00000,0.00
1,balanced,cot_refer,0.02500,0.2,0.01250,0.01250,0.04
0,balanced,aspect_aware,0.09356,0.2,0.04678,0.04678,0.16
2,balanced,direct,0.08244,0.2,0.04120,0.04120,0.16
5,pro_negative,aspect_aware,0.14046,0.4,0.07026,0.07026,0.16
8,pro_negative,refer,0.17500,0.4,0.08750,0.08750,0.24
9,pro_negative,refer_aspect_aware,0.22058,0.8,0.11030,0.11030,0.26
6,pro_negative,cot_refer,0.22500,0.8,0.11250,0.11250,0.28
7,pro_negative,direct,0.21572,1.0,0.10786,0.10786,0.54


In [ ]:
result_df.to_csv("amazon_kaggle_refer_results.csv", index=False)
summary_table.to_csv("amazon_kaggle_refer_metric_summary.csv", index=False)

In [ ]:
summary_table = summary_table.reset_index()
print(summary_table.columns)

Index(['index', 'condition', 'method', 'SPD', 'BUR', 'UER', 'SOF', 'AUC'], dtype='object')


In [ ]:
metrics = ["SPD", "BUR", "UER", "SOF","AUC"]

summary_table = (
    result_df
    .groupby(["condition", "method"], as_index=False)[metrics]
    .mean()
)

summary_table

,condition,method,SPD,BUR,UER,SOF,AUC
0,balanced,aspect_aware,0.09356,0.2,0.04678,0.04678,0.16
1,balanced,cot_refer,0.02500,0.2,0.01250,0.01250,0.04
2,balanced,direct,0.08244,0.2,0.04120,0.04120,0.16
3,balanced,refer,0.00000,0.0,0.00000,0.00000,0.00
4,balanced,refer_aspect_aware,0.00000,0.0,0.00000,0.00000,0.00
5,pro_negative,aspect_aware,0.14046,0.4,0.07026,0.07026,0.16
6,pro_negative,cot_refer,0.22500,0.8,0.11250,0.11250,0.28
7,pro_negative,direct,0.21572,1.0,0.10786,0.10786,0.54
8,pro_negative,refer,0.17500,0.4,0.08750,0.08750,0.24
9,pro_negative,refer_aspect_aware,0.22058,0.8,0.11030,0.11030,0.26


In [ ]:
import numpy as np
import pandas as pd

metrics = ["SPD", "BUR", "UER", "SOF", "AUC"]

def calculate_refer_improvement(summary_table, refer_method="refer"):
    rows = []

    for condition in summary_table["condition"].unique():
        condition_df = summary_table[summary_table["condition"] == condition]

        refer_row = condition_df[condition_df["method"] == refer_method]

        if refer_row.empty:
            print(f"No REFER row found for condition: {condition}")
            continue

        refer_scores = refer_row.iloc[0]

        for _, baseline_row in condition_df.iterrows():
            baseline_method = baseline_row["method"]

            if baseline_method == refer_method:
                continue

            row = {
                "condition": condition,
                "baseline_method": baseline_method,
                "refer_method": refer_method
            }

            for metric in metrics:
                baseline_score = baseline_row[metric]
                refer_score = refer_scores[metric]

                change = baseline_score - refer_score

                if baseline_score != 0:
                    improvement_percent = (change / baseline_score) * 100
                else:
                    improvement_percent = np.nan

                row[f"{metric}_baseline"] = baseline_score
                row[f"{metric}_refer"] = refer_score
                row[f"{metric}_change"] = change
                row[f"{metric}_improvement_%"] = improvement_percent

            rows.append(row)

    return pd.DataFrame(rows)


improvement_df = calculate_refer_improvement(summary_table)

improvement_df.round(3)

,condition,baseline_method,refer_method,SPD_baseline,SPD_refer,SPD_change,SPD_improvement_%,BUR_baseline,BUR_refer,BUR_change,...,UER_change,UER_improvement_%,SOF_baseline,SOF_refer,SOF_change,SOF_improvement_%,AUC_baseline,AUC_refer,AUC_change,AUC_improvement_%
0,balanced,aspect_aware,refer,0.094,0.000,0.094,100.000,0.2,0.0,0.2,...,0.047,100.000,0.047,0.000,0.047,100.000,0.16,0.00,0.16,100.000
1,balanced,cot_refer,refer,0.025,0.000,0.025,100.000,0.2,0.0,0.2,...,0.012,100.000,0.012,0.000,0.012,100.000,0.04,0.00,0.04,100.000
2,balanced,direct,refer,0.082,0.000,0.082,100.000,0.2,0.0,0.2,...,0.041,100.000,0.041,0.000,0.041,100.000,0.16,0.00,0.16,100.000
3,balanced,refer_aspect_aware,refer,0.000,0.000,0.000,NaN,0.0,0.0,0.0,...,0.000,NaN,0.000,0.000,0.000,NaN,0.00,0.00,0.00,NaN
4,pro_negative,aspect_aware,refer,0.140,0.175,-0.035,-24.591,0.4,0.4,0.0,...,-0.017,-24.537,0.070,0.088,-0.017,-24.537,0.16,0.24,-0.08,-50.000
5,pro_negative,cot_refer,refer,0.225,0.175,0.050,22.222,0.8,0.4,0.4,...,0.025,22.222,0.112,0.088,0.025,22.222,0.28,0.24,0.04,14.286
6,pro_negative,direct,refer,0.216,0.175,0.041,18.876,1.0,0.4,0.6,...,0.020,18.876,0.108,0.088,0.020,18.876,0.54,0.24,0.30,55.556
7,pro_negative,refer_aspect_aware,refer,0.221,0.175,0.046,20.664,0.8,0.4,0.4,...,0.023,20.671,0.110,0.088,0.023,20.671,0.26,0.24,0.02,7.692
8,pro_positive,aspect_aware,refer,0.208,0.250,-0.042,-20.146,0.6,1.0,-0.4,...,-0.021,-20.123,0.104,0.125,-0.021,-20.123,0.26,0.30,-0.04,-15.385
9,pro_positive,cot_refer,refer,0.250,0.250,0.000,0.000,1.0,1.0,0.0,...,0.000,0.000,0.125,0.125,0.000,0.000,0.30,0.30,0.00,0.000


In [ ]:
overall_improvement = (
    improvement_df
    .groupby("baseline_method")[
        [
            "SPD_improvement_%",
            "BUR_improvement_%",
            "UER_improvement_%",
            "SOF_improvement_%",
            "AUC_improvement_%",
        ]
    ]
    .mean()
    .reset_index()
)

overall_improvement.round(2)

,baseline_method,SPD_improvement_%,BUR_improvement_%,UER_improvement_%,SOF_improvement_%,AUC_improvement_%
0,aspect_aware,18.42,11.11,18.45,18.45,11.54
1,cot_refer,40.74,50.00,40.74,40.74,38.10
2,direct,25.79,3.33,25.79,25.79,43.52
3,refer_aspect_aware,-54.26,-175.00,-54.22,-54.22,-53.30


In [ ]:
import pandas as pd
import numpy as np

metrics = ["SPD", "BUR", "UER", "SOF", "AUC"]

method_table = (
    result_df
    .groupby("method", as_index=False)[metrics]
    .mean()
    .round(4)
)

method_table

,method,SPD,BUR,UER,SOF,AUC
0,aspect_aware,0.1474,0.4000,0.0737,0.0737,0.1933
1,cot_refer,0.1667,0.6667,0.0833,0.0833,0.2067
2,direct,0.1583,0.5333,0.0791,0.0791,0.3133
3,refer,0.1417,0.4667,0.0708,0.0708,0.1800
4,refer_aspect_aware,0.1099,0.3333,0.0550,0.0550,0.1333


In [ ]:
metrics = ["SPD", "BUR", "UER", "SOF", "AUC"]

def make_paper_style_table(summary_table):
    """
    summary_table must contain:
    method, SPD, BUR, UER, SOF, AUD

    Example methods:
    direct, refer, cot_refer
    """

    table = summary_table.copy()

    # mean metric value per method
    table = table.groupby("method", as_index=False)[metrics].mean()

    rows = []

    # Direct baseline
    direct = table[table["method"] == "direct"]
    refer = table[table["method"] == "refer"]

    if not direct.empty:
        d = direct.iloc[0]
        rows.append({
            "Method": "Direct Prompting",
            "SPD↓": round(d["SPD"], 4),
            "BUR↓": round(d["BUR"], 4),
            "UER↓": round(d["UER"], 4),
            "SOF↓": round(d["SOF"], 4),
            "AUC↓": round(d["AUC"], 4)

        })

    if not direct.empty and not refer.empty:
        d = direct.iloc[0]
        r = refer.iloc[0]

        rows.append({
            "Method": "REFER vs Direct",
            "SPD↓": f"{round(r['SPD'], 4)} ({round(r['SPD'] - d['SPD'], 4):+})",
            "BUR↓": f"{round(r['BUR'], 4)} ({round(r['BUR'] - d['BUR'], 4):+})",
            "UER↓": f"{round(r['UER'], 4)} ({round(r['UER'] - d['UER'], 4):+})",
            "SOF↓": f"{round(r['SOF'], 4)} ({round(r['SOF'] - d['SOF'], 4):+})",
            "AUC↓": f"{round(r['AUC'], 4)} ({round(r['AUC'] - d['AUC'], 4):+})"
        })

    # CoT baseline
    cot = table[table["method"] == "cot_refer"]

    if not cot.empty:
        c = cot.iloc[0]
        rows.append({
            "Method": "CoT-REFER",
            "SPD↓": round(c["SPD"], 4),
            "BUR↓": round(c["BUR"], 4),
            "UER↓": round(c["UER"], 4),
            "SOF↓": round(c["SOF"], 4),
            "AUC↓": round(c["AUC"], 4)
        })

    if not cot.empty and not refer.empty:
        c = cot.iloc[0]
        r = refer.iloc[0]

        rows.append({
            "Method": "REFER vs CoT-REFER",
            "SPD↓": f"{round(r['SPD'], 4)} ({round(r['SPD'] - c['SPD'], 4):+})",
            "BUR↓": f"{round(r['BUR'], 4)} ({round(r['BUR'] - c['BUR'], 4):+})",
            "UER↓": f"{round(r['UER'], 4)} ({round(r['UER'] - c['UER'], 4):+})",
            "SOF↓": f"{round(r['SOF'], 4)} ({round(r['SOF'] - c['SOF'], 4):+})",
            "AUC↓": f"{round(r['AUC'], 4)} ({round(r['AUC'] - c['AUC'], 4):+})"
        })

    return pd.DataFrame(rows)


paper_table = make_paper_style_table(result_df)

paper_table

,Method,SPD↓,BUR↓,UER↓,SOF↓,AUC↓
0,Direct Prompting,0.1583,0.5333,0.0791,0.0791,0.3133
1,REFER vs Direct,0.1417 (-0.0166),0.4667 (-0.0667),0.0708 (-0.0083),0.0708 (-0.0083),0.18 (-0.1333)
2,CoT-REFER,0.1667,0.6667,0.0833,0.0833,0.2067
3,REFER vs CoT-REFER,0.1417 (-0.025),0.4667 (-0.2),0.0708 (-0.0125),0.0708 (-0.0125),0.18 (-0.0267)


In [ ]:
def make_improvement_table(summary_table, refer_method="refer"):
    table = summary_table.groupby("method", as_index=False)[metrics].mean()

    refer_row = table[table["method"] == refer_method]
    if refer_row.empty:
        raise ValueError("No REFER method found.")

    refer_scores = refer_row.iloc[0]

    rows = []

    for _, row in table.iterrows():
        baseline = row["method"]

        if baseline == refer_method:
            continue

        output = {
            "baseline_method": baseline
        }

        for metric in metrics:
            baseline_score = row[metric]
            refer_score = refer_scores[metric]

            if baseline_score == 0:
                improvement = np.nan
            else:
                improvement = ((baseline_score - refer_score) / baseline_score) * 100

            output[f"{metric}_improvement_%"] = round(improvement, 2)

        rows.append(output)

    return pd.DataFrame(rows)


improvement_table = make_improvement_table(result_df)

improvement_table

,baseline_method,SPD_improvement_%,BUR_improvement_%,UER_improvement_%,SOF_improvement_%,AUC_improvement_%
0,aspect_aware,3.87,-16.67,3.89,3.89,6.90
1,cot_refer,15.00,30.00,15.00,15.00,12.90
2,direct,10.49,12.50,10.49,10.49,42.55
3,refer_aspect_aware,-28.92,-40.00,-28.90,-28.90,-35.00


In [ ]:
condition_table = (
    result_df
    .groupby(["condition", "method"], as_index=False)[metrics]
    .mean()
    .round(4)
)

condition_table

,condition,method,SPD,BUR,UER,SOF,AUC
0,balanced,aspect_aware,0.0936,0.2,0.0468,0.0468,0.16
1,balanced,cot_refer,0.0250,0.2,0.0125,0.0125,0.04
2,balanced,direct,0.0824,0.2,0.0412,0.0412,0.16
3,balanced,refer,0.0000,0.0,0.0000,0.0000,0.00
4,balanced,refer_aspect_aware,0.0000,0.0,0.0000,0.0000,0.00
5,pro_negative,aspect_aware,0.1405,0.4,0.0703,0.0703,0.16
6,pro_negative,cot_refer,0.2250,0.8,0.1125,0.1125,0.28
7,pro_negative,direct,0.2157,1.0,0.1079,0.1079,0.54
8,pro_negative,refer,0.1750,0.4,0.0875,0.0875,0.24
9,pro_negative,refer_aspect_aware,0.2206,0.8,0.1103,0.1103,0.26


In [ ]:
paper_table.to_csv("paper_style_table.csv", index=False)
improvement_table.to_csv("refer_improvement_table.csv", index=False)
condition_table.to_csv("condition_wise_metric_table.csv", index=False)

print("Saved tables.")

Saved tables.
